In [2]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

In [ ]:
credits = pd.read_csv('tmdb_5000_credits.csv')
movies = pd.read_csv('tmdb_5000_movies.csv')

print("📊 شكل البيانات:")
print(f"Credits: {credits.shape}")
print(f"Movies: {movies.shape}")

📊 شكل البيانات:
Credits: (4803, 4)
Movies: (4803, 20)


In [ ]:
movies = movies.merge(credits, on='title')

In [ ]:
def convert(obj):
    """تحويل البيانات من JSON string إلى list"""
    if pd.isna(obj):
        return []
    try:
        L = []
        for i in ast.literal_eval(obj):
            L.append(i['name'])
        return L
    except:
        return []

def convert_cast(obj):
    """استخراج أول 3 ممثلين"""
    if pd.isna(obj):
        return []
    try:
        L = []
        counter = 0
        for i in ast.literal_eval(obj):
            if counter < 3:
                L.append(i['name'])
                counter += 1
            else:
                break
        return L
    except:
        return []

def convert_crew(obj):
    """استخراج المخرج فقط"""
    if pd.isna(obj):
        return []
    try:
        L = []
        for i in ast.literal_eval(obj):
            if i['job'] == 'Director':
                L.append(i['name'])
                break
        return L
    except:
        return []

print("🔄 معالجة البيانات...")

🔄 معالجة البيانات...


In [ ]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert_cast)
movies['crew'] = movies['crew'].apply(convert_crew)
movies['overview'] = movies['overview'].fillna('')

In [ ]:
movies['tags'] = movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies['tags'] = movies['tags'] + movies['overview'].apply(lambda x: [x])
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x) if isinstance(x, list) else x)
movies['tags'] = movies['tags'].apply(lambda x: x.lower() if isinstance(x, str) else '')

In [ ]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'vote_average', 'vote_count', 'tags']]
movies = movies.dropna()

print("✅ تمت معالجة البيانات بنجاح!")
print(f"عدد الأفلام بعد التنظيف: {len(movies)}")

✅ تمت معالجة البيانات بنجاح!
عدد الأفلام بعد التنظيف: 4809


In [ ]:
print("\n🎬 بناء نظام التوصية Based-Content...")

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['tags'])
cosine_sim_content = cosine_similarity(tfidf_matrix, tfidf_matrix)

def get_content_recommendations(title, cosine_sim=cosine_sim_content, n_recommendations=10):
    """توصيات Based-Content"""
    try:
        idx = movies[movies['title'] == title].index[0]
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:n_recommendations+1]
        movie_indices = [i[0] for i in sim_scores]
        
        recommendations = movies.iloc[movie_indices][['title', 'vote_average', 'vote_count', 'genres']].copy()
        recommendations['similarity_score'] = [i[1] for i in sim_scores]
        
        return recommendations
    except Exception as e:
        print(f"Error in content recommendations: {e}")
        return None


🎬 بناء نظام التوصية Based-Content...


In [ ]:
np.random.seed(42)
n_users = 1000
n_movies = len(movies)


👥 إنشاء بيانات التقييمات المحاكاة...


In [ ]:
def generate_realistic_ratings(n_users, movies_df):
    ratings_data = []
    
    for user_id in range(1, n_users + 1):
        n_ratings = np.random.randint(20, 100)
        rated_movies = np.random.choice(movies_df['movie_id'].values, n_ratings, replace=False)
        
        for movie_id in rated_movies:
            movie_avg = movies_df[movies_df['movie_id'] == movie_id]['vote_average'].values[0]
            
            base_rating = movie_avg / 2
            rating = np.random.normal(base_rating, 1.0)
            
            rating = max(0.5, min(5.0, rating))
            rating = round(rating * 2) / 2
            
            ratings_data.append({
                'user_id': user_id,
                'movie_id': movie_id,
                'rating': rating
            })
    
    return pd.DataFrame(ratings_data)

ratings_df = generate_realistic_ratings(n_users, movies)
print(f"📈 مصفوفة التقييمات: {ratings_df.shape}")
print(f"عدد التقييمات الفريدة: {ratings_df['user_id'].nunique()} مستخدم × {ratings_df['movie_id'].nunique()} فيلم")

📈 مصفوفة التقييمات: (59360, 3)
عدد التقييمات الفريدة: 1000 مستخدم × 4803 فيلم


In [ ]:
def create_user_movie_matrix(ratings_df):
    """إنشاء مصفوفة المستخدم-العنصر"""
    user_movie_matrix = ratings_df.pivot_table(
        index='user_id', 
        columns='movie_id', 
        values='rating'
    ).fillna(0)
    return user_movie_matrix

user_movie_matrix = create_user_movie_matrix(ratings_df)

def item_based_recommendations(movie_id, user_movie_matrix=user_movie_matrix, n_recommendations=10):
    """توصيات Item-Based Collaborative Filtering"""
    try:
        sparse_matrix = csr_matrix(user_movie_matrix.values)
        item_similarity = cosine_similarity(sparse_matrix.T)
        
        item_similarity_df = pd.DataFrame(
            item_similarity, 
            index=user_movie_matrix.columns, 
            columns=user_movie_matrix.columns
        )
        
        if movie_id not in item_similarity_df.columns:
            return None
            
        similar_scores = item_similarity_df[movie_id].sort_values(ascending=False)
        similar_movies = similar_scores.iloc[1:n_recommendations+1]
        
        recommendations = []
        for similar_movie_id, score in similar_movies.items():
            movie_info = movies[movies['movie_id'] == similar_movie_id]
            if not movie_info.empty:
                title = movie_info['title'].values[0]
                vote_avg = movie_info['vote_average'].values[0]
                genres = movie_info['genres'].values[0]
                recommendations.append({
                    'title': title,
                    'similarity_score': score,
                    'vote_average': vote_avg,
                    'genres': genres,
                    'movie_id': similar_movie_id
                })
        
        return pd.DataFrame(recommendations)
    except Exception as e:
        print(f"Error in item-based recommendations: {e}")
        return None


🔧 بناء نظام الترشيح التعاوني Item-Based...


In [ ]:
def matrix_factorization_recommendations(user_id, user_movie_matrix=user_movie_matrix, n_recommendations=10, n_factors=50):
    """توصيات باستخدام Matrix Factorization (SVD)"""
    try:
        if user_id not in user_movie_matrix.index:
            return None
            
        R = user_movie_matrix.values
        user_ratings_mean = np.mean(R, axis=1)
        R_demeaned = R - user_ratings_mean.reshape(-1, 1)
        
        U, sigma, Vt = svds(R_demeaned, k=min(n_factors, min(R_demeaned.shape)-1))
        sigma = np.diag(sigma)
        
        all_user_predicted_ratings = np.dot(np.dot(U, sigma), Vt) + user_ratings_mean.reshape(-1, 1)
        
        preds_df = pd.DataFrame(
            all_user_predicted_ratings,
            index=user_movie_matrix.index,
            columns=user_movie_matrix.columns
        )
        
        user_row = user_movie_matrix.index.get_loc(user_id)
        user_predictions = preds_df.iloc[user_row]
        
        user_actual_ratings = user_movie_matrix.loc[user_id]
        unwatched_movies = user_predictions[user_actual_ratings == 0]
        
        top_recommendations = unwatched_movies.sort_values(ascending=False).head(n_recommendations)
        
        recommendations = []
        for movie_id, predicted_rating in top_recommendations.items():
            movie_info = movies[movies['movie_id'] == movie_id]
            if not movie_info.empty:
                title = movie_info['title'].values[0]
                actual_rating = movie_info['vote_average'].values[0]
                genres = movie_info['genres'].values[0]
                recommendations.append({
                    'title': title,
                    'predicted_rating': predicted_rating,
                    'actual_rating': actual_rating,
                    'genres': genres,
                    'movie_id': movie_id
                })
        
        return pd.DataFrame(recommendations)
    except Exception as e:
        print(f"Error in matrix factorization: {e}")
        return None


📊 تطبيق تحليل المصفوفة (SVD)...


In [ ]:
print("\n📈 تقييم أداء النظام...")

def precision_at_k(actual_items, recommended_items, k=10):
    """حساب Precision@K"""
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    
    relevant_count = len(set(actual_items) & set(recommended_items))
    return relevant_count / len(recommended_items) if recommended_items else 0

def recall_at_k(actual_items, recommended_items, k=10):
    """حساب Recall@K"""
    if len(recommended_items) > k:
        recommended_items = recommended_items[:k]
    
    relevant_count = len(set(actual_items) & set(recommended_items))
    return relevant_count / len(actual_items) if actual_items else 0

def evaluate_recommendation_system(movies_sample=10, k=5):
    """تقييم شامل لنظام التوصية"""
    print(f"🔍 جاري التقييم على {movies_sample} أفلام...")
    
    sample_movies = movies.sample(min(movies_sample, len(movies)))
    content_precisions = []
    item_precisions = []
    
    for _, movie in sample_movies.iterrows():
        movie_title = movie['title']
        movie_id = movie['movie_id']
        movie_genres = set(movie['genres'])
        
        relevant_movies = []
        for _, other_movie in movies.iterrows():
            if len(set(other_movie['genres']) & movie_genres) >= 1:
                relevant_movies.append(other_movie['title'])
        
        # تقييم Based-Content
        content_recs = get_content_recommendations(movie_title, n_recommendations=k*2)
        if content_recs is not None:
            content_precision = precision_at_k(relevant_movies, content_recs['title'].tolist(), k)
            content_precisions.append(content_precision)
        
        # تقييم Item-Based
        item_recs = item_based_recommendations(movie_id, n_recommendations=k*2)
        if item_recs is not None:
            item_precision = precision_at_k(relevant_movies, item_recs['title'].tolist(), k)
            item_precisions.append(item_precision)
    
    avg_content_precision = np.mean(content_precisions) if content_precisions else 0
    avg_item_precision = np.mean(item_precisions) if item_precisions else 0
    
    print(f"🎯 Based-Content Precision@{k}: {avg_content_precision:.3f}")
    print(f"👥 Item-Based Precision@{k}: {avg_item_precision:.3f}")
    
    return {
        'content_precision': avg_content_precision,
        'item_precision': avg_item_precision
    }


📈 تقييم أداء النظام...


In [ ]:
evaluation_results = evaluate_recommendation_system()

🔍 جاري التقييم على 10 أفلام...
🎯 Based-Content Precision@5: 0.700
👥 Item-Based Precision@5: 0.580


In [ ]:
print("\n🏆 بناء نظام التوصية بالأفلام الأعلى تقييماً...")

def get_top_rated_unseen_movies(user_id, user_movie_matrix, n_recommendations=10):
    """توصية بالأفلام الأعلى تقييماً التي لم يشاهدها المستخدم"""
    try:
        if user_id not in user_movie_matrix.index:
            return None
            
        C = movies['vote_average'].mean()
        m = movies['vote_count'].quantile(0.8)
        
        qualified_movies = movies.copy().loc[movies['vote_count'] >= m]
        
        def weighted_rating(x, m=m, C=C):
            v = x['vote_count']
            R = x['vote_average']
            return (v/(v+m) * R) + (m/(m+v) * C)
        
        qualified_movies['score'] = qualified_movies.apply(weighted_rating, axis=1)
        qualified_movies = qualified_movies.sort_values('score', ascending=False)
        
        user_ratings = user_movie_matrix.loc[user_id]
        watched_movies = user_ratings[user_ratings > 0].index
        unseen_movies = qualified_movies[~qualified_movies['movie_id'].isin(watched_movies)]
        
        return unseen_movies[['title', 'score', 'vote_average', 'vote_count', 'genres', 'movie_id']].head(n_recommendations)
    
    except Exception as e:
        print(f"Error in top rated recommendations: {e}")
        return None


🏆 بناء نظام التوصية بالأفلام الأعلى تقييماً...


In [ ]:
print("\n🧪 اختبار جميع أنظمة التوصية...")

test_movie = "The Dark Knight Rises"
test_user_id = 1

print(f"\n🎯 اختبار Based-Content لفيلم '{test_movie}':")
content_recs = get_content_recommendations(test_movie)
if content_recs is not None:
    print(content_recs[['title', 'similarity_score']].head())

print(f"\n👥 اختبار Item-Based لفيلم '{test_movie}':")
test_movie_id = movies[movies['title'] == test_movie]['movie_id'].values[0]
item_recs = item_based_recommendations(test_movie_id)
if item_recs is not None:
    print(item_recs[['title', 'similarity_score']].head())

print(f"\n📊 اختبار Matrix Factorization للمستخدم {test_user_id}:")
mf_recs = matrix_factorization_recommendations(test_user_id)
if mf_recs is not None:
    print(mf_recs[['title', 'predicted_rating']].head())

print(f"\n🏆 اختبار الأفلام الأعلى تقييماً للمستخدم {test_user_id}:")
top_rated_recs = get_top_rated_unseen_movies(test_user_id, user_movie_matrix)
if top_rated_recs is not None:
    print(top_rated_recs[['title', 'score']].head())


🧪 اختبار جميع أنظمة التوصية...

🎯 اختبار Based-Content لفيلم 'The Dark Knight Rises':
                title  similarity_score
65    The Dark Knight          0.508981
119     Batman Begins          0.433882
428    Batman Returns          0.427200
1361           Batman          0.400396
1360           Batman          0.396697

👥 اختبار Item-Based لفيلم 'The Dark Knight Rises':
                title  similarity_score
0             Ed Wood          0.310941
1        Undiscovered          0.261724
2             Beastly          0.257261
3  Friends with Money          0.240730
4            Coraline          0.223695

📊 اختبار Matrix Factorization للمستخدم 1:
                         title  predicted_rating
0         (500) Days of Summer          0.420035
1  Once Upon a Time in America          0.419943
2             A Beautiful Mind          0.413478
3                      Caramel          0.411232
4               The Last Waltz          0.409658

🏆 اختبار الأفلام الأعلى تقييماً للمستخدم 1:

In [ ]:
data_to_save = {
    'movies': movies,
    'cosine_sim_content': cosine_sim_content,
    'user_movie_matrix': user_movie_matrix,
    'ratings_df': ratings_df,
    'tfidf_vectorizer': tfidf
}

with open('movie_recommendation_data.pkl', 'wb') as f:
    pickle.dump(data_to_save, f)


💾 حفظ النماذج والبيانات...


In [ ]:
recommendation_config = {
    'content_based': 'get_content_recommendations',
    'item_based': 'item_based_recommendations', 
    'matrix_factorization': 'matrix_factorization_recommendations',
    'top_rated_unseen': 'get_top_rated_unseen_movies'
}

with open('recommendation_config.pkl', 'wb') as f:
    pickle.dump(recommendation_config, f)

print("✅ تم حفظ جميع البيانات والنماذج بنجاح!")

✅ تم حفظ جميع البيانات والنماذج بنجاح!


In [ ]:
print("\n📈 إحصائيات النظام النهائية:")
print(f"• عدد الأفلام: {len(movies)}")
print(f"• عدد المستخدمين: {len(user_movie_matrix)}")
print(f"• عدد التقييمات: {len(ratings_df)}")
print(f"• متوسط التقييم: {movies['vote_average'].mean():.2f}")
print(f"• متوسط Precision@5: {evaluation_results['content_precision']:.3f}")

print("\n🎉 اكتمل بناء نظام التوصية المتقدم بنجاح!")


📈 إحصائيات النظام النهائية:
• عدد الأفلام: 4809
• عدد المستخدمين: 1000
• عدد التقييمات: 59360
• متوسط التقييم: 6.09
• متوسط Precision@5: 0.700

🎉 اكتمل بناء نظام التوصية المتقدم بنجاح!
